# Notebook 03: Advanced Tuning + Experiment Tracking

Goal: teach repeatable experimentation and hyperparameter tuning.


In [3]:
import json
import logging
import sys
import time
from contextlib import contextmanager
from datetime import datetime
from pathlib import Path


def _project_root() -> Path:
    cwd = Path.cwd().resolve()
    if (cwd / "src").is_dir():
        return cwd
    parent = cwd.parent
    if (parent / "src").is_dir():
        return parent
    return cwd


PROJECT_ROOT = _project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

from src.config import PROCESSED_DIR
from src.data import load_raw_data, basic_cleaning, make_text_feature, split_data
from src.features import build_text_tabular_preprocessor

LOG_DIR = PROJECT_ROOT / 'logs/notebooks'
LOG_DIR.mkdir(parents=True, exist_ok=True)
LOG_FILE = LOG_DIR / f"03_advanced_tuning_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"

logger = logging.getLogger('nb03')
logger.setLevel(logging.INFO)
logger.handlers.clear()
_fmt = logging.Formatter('%(asctime)s | %(levelname)s | %(message)s')
_fh = logging.FileHandler(LOG_FILE, encoding='utf-8')
_fh.setFormatter(_fmt)
logger.addHandler(_fh)

@contextmanager
def timed_step(name: str):
    start = time.perf_counter()
    logger.info('START: %s', name)
    print(f"[START] {name}")
    try:
        yield
    finally:
        elapsed = time.perf_counter() - start
        logger.info('END: %s | elapsed=%.2fs', name, elapsed)
        print(f"[END] {name} | elapsed={elapsed:.2f}s")


def get_xgb_config():
    """Use GPU when available; otherwise fall back to CPU."""
    gpu_cfg = {'tree_method': 'hist', 'device': 'cuda'}
    cpu_cfg = {'tree_method': 'hist', 'device': 'cpu'}
    try:
        probe = XGBClassifier(n_estimators=1, eval_metric='logloss', **gpu_cfg)
        X_probe = np.array([[0.0], [1.0]])
        y_probe = np.array([0, 1])
        probe.fit(X_probe, y_probe)
        logger.info('GPU detected and enabled for XGBoost.')
        print('[INFO] GPU detected and enabled for XGBoost')
        return gpu_cfg
    except Exception as e:
        logger.info('GPU unavailable, falling back to CPU. reason=%s', e)
        print('[INFO] GPU unavailable, falling back to CPU')
        return cpu_cfg

XGB_ACCEL = get_xgb_config()
print('project root:', PROJECT_ROOT)
print('notebook log file:', LOG_FILE)


[INFO] GPU detected and enabled for XGBoost
project root: C:\Users\akhil\OneDrive\Documents\ML Kaggle DS Projects\womens-e-commerce-reviews
notebook log file: C:\Users\akhil\OneDrive\Documents\ML Kaggle DS Projects\womens-e-commerce-reviews\logs\notebooks\03_advanced_tuning_20260426_095057.log


In [4]:
with timed_step('load_clean_split_data'):
    df = load_raw_data()
    df = basic_cleaning(df)
    df['text'] = make_text_feature(df)
    X_train, X_valid, y_train, y_valid = split_data(df)

logger.info('train_shape=%s valid_shape=%s', X_train.shape, X_valid.shape)
print(X_train.shape, X_valid.shape)


[START] load_clean_split_data
[END] load_clean_split_data | elapsed=0.19s
(18788, 11) (4698, 11)


## Baseline advanced model score (CV)


In [5]:
base_pipeline = Pipeline([
    ('prep', build_text_tabular_preprocessor(text_col='text')),
    ('model', XGBClassifier(
        n_estimators=300,
        max_depth=10,
        learning_rate=0.1,
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=42,
        eval_metric='logloss',
        **XGB_ACCEL,
    ))
])

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
with timed_step('baseline_cv_score'):
    base_cv_f1 = cross_val_score(base_pipeline, X_train, y_train, scoring='f1', cv=cv, n_jobs=1)

logger.info('base_cv_f1_mean=%.6f base_cv_f1_std=%.6f', base_cv_f1.mean(), base_cv_f1.std())
print('Base CV F1 mean:', base_cv_f1.mean())
print('Base CV F1 std :', base_cv_f1.std())


[START] baseline_cv_score


c:\Users\akhil\OneDrive\Documents\ML Kaggle DS Projects\womens-e-commerce-reviews\.venv\Lib\site-packages\xgboost\core.py:751: UserWarning: [09:51:28] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


[END] baseline_cv_score | elapsed=97.90s
Base CV F1 mean: 0.9634788491179459
Base CV F1 std : 0.0010625733605671413


## Optuna tuning loop


In [6]:
import optuna

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 200, 700, step=100),
        'max_depth': trial.suggest_int('max_depth', 6, 14),
        'learning_rate': trial.suggest_float('learning_rate', 0.03, 0.2, log=True),
        'subsample': trial.suggest_float('subsample', 0.7, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 1.0),
    }

    pipe = Pipeline([
        ('prep', build_text_tabular_preprocessor(text_col='text')),
        ('model', XGBClassifier(
            **params,
            random_state=42,
            eval_metric='logloss',
            **XGB_ACCEL,
        ))
    ])

    score = cross_val_score(pipe, X_train, y_train, scoring='f1', cv=cv, n_jobs=1).mean()
    logger.info('trial=%s score=%.6f params=%s', trial.number, score, params)
    return score

study = optuna.create_study(direction='maximize')
with timed_step('optuna_tuning'):
    study.optimize(objective, n_trials=10)

logger.info('best_score=%.6f best_params=%s', study.best_value, study.best_trial.params)
print('Best score:', study.best_value)
print('Best params:', study.best_trial.params)


c:\Users\akhil\OneDrive\Documents\ML Kaggle DS Projects\womens-e-commerce-reviews\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-04-26 09:52:35,301] A new study created in memory with name: no-name-f7130e40-64f6-45fd-94dc-5b6e75bac6c6


[START] optuna_tuning


[I 2026-04-26 09:54:49,721] Trial 0 finished with value: 0.9627342399173845 and parameters: {'n_estimators': 500, 'max_depth': 10, 'learning_rate': 0.10200398102325207, 'subsample': 0.7800270701487998, 'colsample_bytree': 0.8467945139801369}. Best is trial 0 with value: 0.9627342399173845.
[I 2026-04-26 09:59:08,991] Trial 1 finished with value: 0.9637041405208548 and parameters: {'n_estimators': 700, 'max_depth': 13, 'learning_rate': 0.04124486360468805, 'subsample': 0.8191816478021546, 'colsample_bytree': 0.9417612867892543}. Best is trial 1 with value: 0.9637041405208548.
[I 2026-04-26 10:00:33,319] Trial 2 finished with value: 0.9635707756057563 and parameters: {'n_estimators': 200, 'max_depth': 13, 'learning_rate': 0.16220633285520925, 'subsample': 0.9306818925354459, 'colsample_bytree': 0.9308497581695124}. Best is trial 1 with value: 0.9637041405208548.
[I 2026-04-26 10:04:02,486] Trial 3 finished with value: 0.9635211467562851 and parameters: {'n_estimators': 600, 'max_depth': 

[END] optuna_tuning | elapsed=1376.54s
Best score: 0.9642432336233439
Best params: {'n_estimators': 700, 'max_depth': 6, 'learning_rate': 0.036327316614859356, 'subsample': 0.7462333553162644, 'colsample_bytree': 0.904679053248568}


In [7]:
# Track experiments in a simple JSONL log
exp_dir = PROCESSED_DIR / 'experiments'
exp_dir.mkdir(parents=True, exist_ok=True)
log_file = exp_dir / 'tuning_runs.jsonl'

record = {
    'timestamp': datetime.now().isoformat(timespec='seconds'),
    'dataset_rows': int(len(df)),
    'target': 'Recommended IND',
    'model': 'XGBClassifier + TFIDF + tabular',
    'accelerator': XGB_ACCEL,
    'best_f1_cv': float(study.best_value),
    'best_params': study.best_trial.params,
    'notebook_log_file': str(LOG_FILE),
}

with timed_step('write_experiment_record'):
    with open(log_file, 'a', encoding='utf-8') as f:
        f.write(json.dumps(record) + '\n')

logger.info('experiment_record=%s', record)
print('Logged experiment to:', log_file)
record


[START] write_experiment_record
[END] write_experiment_record | elapsed=0.00s
Logged experiment to: C:\Users\akhil\OneDrive\Documents\ML Kaggle DS Projects\womens-e-commerce-reviews\data\processed\experiments\tuning_runs.jsonl


{'timestamp': '2026-04-26T10:15:31',
 'dataset_rows': 23486,
 'target': 'Recommended IND',
 'model': 'XGBClassifier + TFIDF + tabular',
 'accelerator': {'tree_method': 'hist', 'device': 'cuda'},
 'best_f1_cv': 0.9642432336233439,
 'best_params': {'n_estimators': 700,
  'max_depth': 6,
  'learning_rate': 0.036327316614859356,
  'subsample': 0.7462333553162644,
  'colsample_bytree': 0.904679053248568},
 'notebook_log_file': 'C:\\Users\\akhil\\OneDrive\\Documents\\ML Kaggle DS Projects\\womens-e-commerce-reviews\\logs\\notebooks\\03_advanced_tuning_20260426_095057.log'}

## Next experiments for students

- Compare RandomForest vs LightGBM.
- Add threshold tuning for F1 optimization.
- Create a train/validation drift check report.
- Track all runs and choose final model by robust metrics.
